# 03. Build Quinquennial Panel

This notebook merges Fraser EFW data with World Bank outcomes to create 
the master quinquennial panel for analysis.

**Inputs:**
- `data/02_intermediate/efw_panel_raw.parquet`
- `data/02_intermediate/worldbank_raw.parquet`

**Outputs:**
- `data/03_clean/panel_quinquennial.parquet`

**Key steps:**
1. Load and merge EFW and World Bank data
2. Validate merge quality
3. Create predetermined controls (lagged values)
4. Create region/income group indicators
5. Final validation and output

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Project paths
PROJECT_ROOT = Path(__file__).parent.parent if '__file__' in dir() else Path.cwd().parent
DATA_INTERMEDIATE = PROJECT_ROOT / 'data/02_intermediate'
DATA_CLEAN = PROJECT_ROOT / 'data/03_clean'

# Ensure output directory exists
DATA_CLEAN.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/gabrielsaco/Documents/GitHub/economic-freedom


## 1. Load Input Data

In [2]:
# Load EFW data
efw_path = DATA_INTERMEDIATE / 'efw_panel_raw.parquet'
assert efw_path.exists(), f"EFW data not found: {efw_path}"
df_efw = pd.read_parquet(efw_path)
print(f"EFW data: {len(df_efw)} observations, {df_efw['iso3c'].nunique()} countries")

# Load World Bank data
wb_path = DATA_INTERMEDIATE / 'worldbank_raw.parquet'
assert wb_path.exists(), f"World Bank data not found: {wb_path}"
df_wb = pd.read_parquet(wb_path)
print(f"World Bank data: {len(df_wb)} observations, {df_wb['iso3c'].nunique()} countries")

# Check year overlap
efw_years = set(df_efw['year'].unique())
wb_years = set(df_wb['year'].unique())
common_years = sorted(efw_years & wb_years)
print(f"\nCommon quinquennial years: {common_years}")

EFW data: 1815 observations, 165 countries
World Bank data: 2405 observations, 219 countries

Common quinquennial years: [np.int64(1970), np.int64(1975), np.int64(1980), np.int64(1985), np.int64(1990), np.int64(1995), np.int64(2000), np.int64(2005), np.int64(2010), np.int64(2015), np.int64(2020)]


## 2. Merge Datasets

In [3]:
# Select columns from each dataset
efw_cols = [
    'iso3c', 'country_name', 'year',
    'efw_aggregate', 'efw_area1', 'efw_area2', 'efw_area3', 'efw_area4', 'efw_area5',
    'd_efw_aggregate', 'd_efw_area1', 'd_efw_area2', 'd_efw_area3', 'd_efw_area4', 'd_efw_area5',
    'efw_sd', 'wb_region', 'wb_income'
]

wb_cols = [
    'iso3c', 'year',
    'gdp_pc_constant', 'gdp_pc_ppp', 'population',
    'ln_gdp_pc', 'growth'
]

# Filter to desired columns (check which exist)
efw_cols_exist = [c for c in efw_cols if c in df_efw.columns]
wb_cols_exist = [c for c in wb_cols if c in df_wb.columns]

df_efw_subset = df_efw[efw_cols_exist].copy()
df_wb_subset = df_wb[wb_cols_exist].copy()

# Merge on country-year
df_panel = df_efw_subset.merge(
    df_wb_subset,
    on=['iso3c', 'year'],
    how='left',
    validate='one_to_one'
)

print(f"Merged panel: {len(df_panel)} observations")
print(f"Countries: {df_panel['iso3c'].nunique()}")

# ASSERTION: No duplicate keys after merge
assert not df_panel.duplicated(subset=['iso3c', 'year']).any(), \
    "CRITICAL: Duplicate country-year keys after merge!"

Merged panel: 1815 observations
Countries: 165


## 3. Merge Quality Assessment

In [4]:
# Check merge success rates
n_total = len(df_panel)
n_gdp = df_panel['gdp_pc_constant'].notna().sum()
n_growth = df_panel['growth'].notna().sum()

print("Merge quality:")
print(f"  Total observations: {n_total}")
print(f"  With GDP per capita: {n_gdp} ({100*n_gdp/n_total:.1f}%)")
print(f"  With growth: {n_growth} ({100*n_growth/n_total:.1f}%)")

# Coverage by year
print("\nMerge by year:")
for year in sorted(df_panel['year'].unique()):
    yr_data = df_panel[df_panel['year'] == year]
    n_efw = len(yr_data)
    n_gdp_yr = yr_data['gdp_pc_constant'].notna().sum()
    print(f"  {year}: {n_efw} countries with EFW, {n_gdp_yr} with GDP ({100*n_gdp_yr/n_efw:.0f}%)")

# Identify unmatched countries
unmatched = df_panel[df_panel['gdp_pc_constant'].isna()]['iso3c'].unique()
if len(unmatched) > 0:
    print(f"\nCountries without World Bank match (sample): {list(unmatched[:10])}")

Merge quality:
  Total observations: 1815
  With GDP per capita: 1661 (91.5%)
  With growth: 1497 (82.5%)

Merge by year:
  1970: 165 countries with EFW, 123 with GDP (75%)
  1975: 165 countries with EFW, 126 with GDP (76%)
  1980: 165 countries with EFW, 134 with GDP (81%)
  1985: 165 countries with EFW, 139 with GDP (84%)
  1990: 165 countries with EFW, 161 with GDP (98%)
  1995: 165 countries with EFW, 162 with GDP (98%)
  2000: 165 countries with EFW, 163 with GDP (99%)
  2005: 165 countries with EFW, 163 with GDP (99%)
  2010: 165 countries with EFW, 163 with GDP (99%)
  2015: 165 countries with EFW, 164 with GDP (99%)
  2020: 165 countries with EFW, 163 with GDP (99%)

Countries without World Bank match (sample): ['AGO', 'ALB', 'ARM', 'AZE', 'BGR', 'BIH', 'BLR', 'BRN', 'COM', 'CPV']


## 4. Create Predetermined Controls (Lagged Variables)

**CRITICAL TIMING RULE:** Controls must be predetermined (lagged) to avoid
post-treatment bias. We use $t-1$ values (prior quinquennial period).

In [5]:
# Sort for lagging
df_panel = df_panel.sort_values(['iso3c', 'year'])

# Lagged outcome variables (predetermined)
df_panel['ln_gdp_pc_lag1'] = df_panel.groupby('iso3c')['ln_gdp_pc'].shift(1)
df_panel['growth_lag1'] = df_panel.groupby('iso3c')['growth'].shift(1)

# Lagged EFW levels (predetermined)
df_panel['efw_aggregate_lag1'] = df_panel.groupby('iso3c')['efw_aggregate'].shift(1)

# Lagged EFW areas (for bundle analysis)
for area in range(1, 6):
    col = f'efw_area{area}'
    df_panel[f'{col}_lag1'] = df_panel.groupby('iso3c')[col].shift(1)

# Verify lags are properly computed
df_panel['year_lag1'] = df_panel.groupby('iso3c')['year'].shift(1)
df_panel['lag_diff'] = df_panel['year'] - df_panel['year_lag1']

non_missing_lags = df_panel['lag_diff'].dropna()
assert (non_missing_lags == 5).all(), "Lag differences are not all 5 years"
print("✓ All lagged variables use proper 5-year lags")

# Clean up
df_panel = df_panel.drop(columns=['year_lag1', 'lag_diff'])

✓ All lagged variables use proper 5-year lags


## 5. Create Lead Variables (for Maintenance Rule Classification ONLY)

**CRITICAL:** These leads are used ONLY for classifying sustained reforms.
They must NEVER be used as controls or in outcome construction.

In [6]:
# Lead EFW values for maintenance rule classification
df_panel['efw_aggregate_lead1'] = df_panel.groupby('iso3c')['efw_aggregate'].shift(-1)
df_panel['efw_aggregate_lead2'] = df_panel.groupby('iso3c')['efw_aggregate'].shift(-2)

print("✓ Created lead variables for maintenance rule classification")
print("  WARNING: These are for classification ONLY, not for controls/outcomes")

✓ Created lead variables for maintenance rule classification


## 6. Create Region and Income Indicators

In [7]:
# Clean region and income classification
df_panel['region'] = df_panel['wb_region'].fillna('Unknown')
df_panel['income_group'] = df_panel['wb_income'].fillna('Unknown')

# Create region dummies for region×time FE
regions = df_panel['region'].unique()
print(f"Regions: {list(regions)}")

# Income group summary
print("\nIncome group distribution:")
print(df_panel.groupby('income_group')['iso3c'].nunique())

Regions: ['Unknown', 'Sub-Saharan Africa', 'Europe & Central Asia', 'Middle East & North Africa', 'Latin America & the Caribbean', 'East Asia & Pacific', 'South Asia', 'North America']

Income group distribution:
income_group
..                       2
High Income             52
Low Income              62
Lower-Middle Income     92
Unknown                165
Upper-Middle Income     71
Name: iso3c, dtype: int64


## 7. Create Initial Condition Variables (for State Dependence)

In [8]:
# Initial log GDP per capita (first available observation per country)
initial_gdp = df_panel.groupby('iso3c')['ln_gdp_pc'].transform('first')
df_panel['ln_gdp_pc_initial'] = initial_gdp

# Initial EFW (first available observation per country)
initial_efw = df_panel.groupby('iso3c')['efw_aggregate'].transform('first')
df_panel['efw_initial'] = initial_efw

# Create quartile bins for state dependence analysis
df_panel['gdp_quartile'] = pd.qcut(
    df_panel['ln_gdp_pc_lag1'].dropna(), 
    q=4, 
    labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)']
).reindex(df_panel.index)

df_panel['efw_quartile'] = pd.qcut(
    df_panel['efw_aggregate_lag1'].dropna(),
    q=4,
    labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)']
).reindex(df_panel.index)

print("✓ Created initial condition and quartile variables for state dependence")

✓ Created initial condition and quartile variables for state dependence


## 8. Summary Statistics

In [9]:
# Key variable summary
key_vars = [
    'efw_aggregate', 'd_efw_aggregate',
    'ln_gdp_pc', 'growth',
    'gdp_pc_constant', 'population'
]

existing_vars = [v for v in key_vars if v in df_panel.columns]

print("Summary Statistics - Quinquennial Panel")
print("="*60)
print(df_panel[existing_vars].describe().round(3))

# EFW areas summary
print("\nEFW Areas Summary:")
efw_area_vars = [f'efw_area{i}' for i in range(1, 6)]
print(df_panel[efw_area_vars].describe().round(2))

# Change variables summary
print("\nEFW Change Variables (Δ):")
d_efw_vars = [f'd_efw_area{i}' for i in range(1, 6)] + ['d_efw_aggregate']
existing_d_vars = [v for v in d_efw_vars if v in df_panel.columns]
print(df_panel[existing_d_vars].describe().round(3))

Summary Statistics - Quinquennial Panel
       efw_aggregate  d_efw_aggregate  ln_gdp_pc    growth  gdp_pc_constant  \
count       1476.000         1311.000   1661.000  1497.000         1661.000   
mean           6.130            0.127      8.294     0.074        11103.688   
std            1.303            0.501      1.508     0.190        16454.674   
min            2.221           -1.938      4.936    -1.250          139.275   
25%            5.224           -0.149      7.094    -0.008         1204.919   
50%            6.160            0.087      8.228     0.083         3742.710   
75%            7.110            0.377      9.558     0.167        14161.146   
max            9.199            2.845     11.671     1.251       117073.212   

         population  
count  1.804000e+03  
mean   3.424435e+07  
std    1.243744e+08  
min    5.360000e+04  
25%    2.683299e+06  
50%    7.202934e+06  
75%    2.213130e+07  
max    1.411100e+09  

EFW Areas Summary:
       efw_area1  efw_area2  e

## 9. Final Validation

In [10]:
print("\n" + "="*60)
print("FINAL VALIDATION CHECKS")
print("="*60)

# 1. No duplicate keys
assert not df_panel.duplicated(subset=['iso3c', 'year']).any(), "Duplicate keys found"
print("✓ No duplicate country-year keys")

# 2. All quinquennial years
QUINQUENNIAL_YEARS = [1970, 1975, 1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020]
assert df_panel['year'].isin(QUINQUENNIAL_YEARS).all(), "Non-quinquennial years found"
print("✓ All years are quinquennial")

# 3. Sufficient panel size
n_obs = len(df_panel)
n_countries = df_panel['iso3c'].nunique()
n_years = df_panel['year'].nunique()
assert n_obs > 1000, "Too few observations"
assert n_countries > 100, "Too few countries"
print(f"✓ Panel size: {n_obs} obs, {n_countries} countries, {n_years} years")

# 4. Outcome coverage
n_with_outcome = df_panel['growth'].notna().sum()
assert n_with_outcome > 800, "Too few observations with growth data"
print(f"✓ Observations with growth outcome: {n_with_outcome}")

# 5. EFW change coverage
n_with_change = df_panel['d_efw_aggregate'].notna().sum()
print(f"✓ Observations with EFW changes: {n_with_change}")


FINAL VALIDATION CHECKS
✓ No duplicate country-year keys
✓ All years are quinquennial
✓ Panel size: 1815 obs, 165 countries, 11 years
✓ Observations with growth outcome: 1497
✓ Observations with EFW changes: 1311


## 10. Save Output

In [11]:
# Save final panel
output_path = DATA_CLEAN / 'panel_quinquennial.parquet'
df_panel.to_parquet(output_path, index=False)
print(f"\n✓ Saved quinquennial panel to {output_path}")
print(f"  Observations: {len(df_panel)}")
print(f"  Countries: {df_panel['iso3c'].nunique()}")
print(f"  Variables: {len(df_panel.columns)}")

# Save variable documentation
var_docs = {
    'identifiers': ['iso3c', 'country_name', 'year'],
    'efw_levels': ['efw_aggregate'] + [f'efw_area{i}' for i in range(1, 6)] + ['efw_sd'],
    'efw_changes': ['d_efw_aggregate'] + [f'd_efw_area{i}' for i in range(1, 6)],
    'outcomes': ['gdp_pc_constant', 'gdp_pc_ppp', 'ln_gdp_pc', 'growth', 'population'],
    'lagged_controls': ['ln_gdp_pc_lag1', 'growth_lag1', 'efw_aggregate_lag1'] + 
                       [f'efw_area{i}_lag1' for i in range(1, 6)],
    'leads_for_classification': ['efw_aggregate_lead1', 'efw_aggregate_lead2'],
    'state_dependence': ['ln_gdp_pc_initial', 'efw_initial', 'gdp_quartile', 'efw_quartile'],
    'groups': ['region', 'income_group', 'wb_region', 'wb_income'],
}

doc_path = DATA_CLEAN / 'panel_documentation.json'
with open(doc_path, 'w') as f:
    json.dump(var_docs, f, indent=2)
print(f"✓ Saved variable documentation to {doc_path}")


✓ Saved quinquennial panel to /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/03_clean/panel_quinquennial.parquet
  Observations: 1815
  Countries: 165
  Variables: 39
✓ Saved variable documentation to /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/03_clean/panel_documentation.json


## 11. Column Summary

In [12]:
print("\n" + "="*60)
print("COLUMN SUMMARY")
print("="*60)
print(f"\nTotal columns: {len(df_panel.columns)}")
for category, cols in var_docs.items():
    existing = [c for c in cols if c in df_panel.columns]
    print(f"\n{category}:")
    for col in existing:
        n_valid = df_panel[col].notna().sum() if df_panel[col].dtype != 'object' else len(df_panel[df_panel[col].notna()])
        print(f"  {col}: {n_valid} non-missing")

print("\n" + "="*60)
print("03_BUILD_QUINQUENNIAL_PANEL COMPLETE")
print("="*60)


COLUMN SUMMARY

Total columns: 39

identifiers:
  iso3c: 1815 non-missing
  country_name: 1815 non-missing
  year: 1815 non-missing

efw_levels:
  efw_aggregate: 1476 non-missing
  efw_area1: 1573 non-missing
  efw_area2: 1716 non-missing
  efw_area3: 1507 non-missing
  efw_area4: 1298 non-missing
  efw_area5: 1583 non-missing
  efw_sd: 1476 non-missing

efw_changes:
  d_efw_aggregate: 1311 non-missing
  d_efw_area1: 1408 non-missing
  d_efw_area2: 1551 non-missing
  d_efw_area3: 1342 non-missing
  d_efw_area4: 1133 non-missing
  d_efw_area5: 1418 non-missing

outcomes:
  gdp_pc_constant: 1661 non-missing
  gdp_pc_ppp: 1125 non-missing
  ln_gdp_pc: 1661 non-missing
  growth: 1497 non-missing
  population: 1804 non-missing

lagged_controls:
  ln_gdp_pc_lag1: 1498 non-missing
  growth_lag1: 1334 non-missing
  efw_aggregate_lag1: 1311 non-missing
  efw_area1_lag1: 1408 non-missing
  efw_area2_lag1: 1551 non-missing
  efw_area3_lag1: 1342 non-missing
  efw_area4_lag1: 1133 non-missing
  e